# Virus H3 coverage maps

Interactive Folium maps of **H3 resolution-4** cells occupied by virus detections in `data/02_interim/viruses_normalized.parquet`.

**What hexes mean:** a cell is drawn if the extract has ≥1 dated detection with an assigned H3 cell there (after optional filters). Only rows with non-null `h3_cell`, `year`, and `month` are mapped — country-only / undated records stay off the map. Empty space is *not* proven pathogen absence.

**Metrics are stored per `(h3_cell, pathogen_name)`:**
- `n_year_months` — how often detections appear (distinct year×month)
- `n_years` / `year_span` — how long the record spans
- `n_records` — raw detection row count (effort-sensitive)

**Map UX:** one polygon per H3 cell on an OpenStreetMap street basemap (hex fill is semi-transparent). Fill color uses the **max** of the plotted metric across pathogens in that cell (shared color scale).

**Hover or click** a hex for a **horizontal** table (metric rows × one column per pathogen). When several pathogens are present, the **layer control** (top-right) toggles pathogens only (H3 polygons stay always on): unchecked pathogens disappear from the popup, and hexes that have **no remaining active pathogen** are hidden.

Set `PATHOGEN_NAME` to filter to one pathogen, or `None` for all pathogens.

After pulling code changes: **restart the kernel** and re-run from the top so `vector_guard.viz` reloads.

Edit the **config** cell (`COUNTRY_ISO2`, `PATHOGEN_NAME`) and re-run from there to redraw.

In [ ]:
from pathlib import Path

import pandas as pd

from vector_guard.data import default_interim_dir, find_repo_root
from vector_guard.viz import aggregate_interim_h3_cells, folium_h3_choropleth

REPO_ROOT = find_repo_root(Path.cwd())
VIRUS_PATH = default_interim_dir(REPO_ROOT) / "viruses_normalized.parquet"
viruses = pd.read_parquet(VIRUS_PATH)
print(f"Loaded {len(viruses):,} rows from {VIRUS_PATH.relative_to(REPO_ROOT)}")
print("With H3:", int(viruses["h3_cell"].notna().sum()), f"/ {len(viruses):,}")
print("Countries:", viruses["country_iso2"].nunique())
print("Pathogens:", sorted(viruses["pathogen_name"].dropna().unique()))

## Config

Set `COUNTRY_ISO2` to an ISO-2 code (e.g. `"ES"`) to filter, or `None` for all countries.  
Set `PATHOGEN_NAME` to a pathogen string (e.g. `"West Nile virus"`) for a single pathogen, or `None` to include all pathogens (multi-pathogen cells list every pathogen in the click popup).

In [ ]:
COUNTRY_ISO2 = None  # e.g. "ES"; None = all countries
PATHOGEN_NAME = None  # e.g. "West Nile virus"; None = all pathogens

In [ ]:
filtered = viruses.copy()
if COUNTRY_ISO2 is not None:
    code = str(COUNTRY_ISO2).strip().upper()
    filtered = filtered.loc[filtered["country_iso2"].astype(str).str.upper() == code]
if PATHOGEN_NAME is not None:
    name = str(PATHOGEN_NAME).strip()
    filtered = filtered.loc[filtered["pathogen_name"].astype(str) == name]

cell_stats = aggregate_interim_h3_cells(
    viruses,
    group_col="pathogen_name",
    country_iso2=COUNTRY_ISO2,
    group_value=PATHOGEN_NAME,
)

n_cells = int(cell_stats["h3_cell"].nunique()) if not cell_stats.empty else 0
n_multi = (
    int((cell_stats.groupby("h3_cell").size() >= 2).sum()) if not cell_stats.empty else 0
)

print(
    f"Filter: country={COUNTRY_ISO2!r}, pathogen_name={PATHOGEN_NAME!r}"
)
print(f"Rows after filter: {len(filtered):,} / {len(viruses):,}")
print(f"(h3_cell, pathogen_name) units: {len(cell_stats):,}")
print(f"Unique H3 cells: {n_cells:,}")
print(f"H3 cells with ≥2 pathogens: {n_multi:,}")
if not cell_stats.empty:
    print("Units by pathogen_name:")
    display(cell_stats["pathogen_name"].value_counts())
    display(
        cell_stats[["n_year_months", "n_years", "year_span", "n_records"]].describe()
    )

## Frequency map (`n_year_months`)

Fill color = **max** `n_year_months` across pathogens in the cell. Hover/click a hex for a horizontal stats table (one column per pathogen).

In [ ]:
freq_title = "n_year_months"
if COUNTRY_ISO2:
    freq_title += f" · {str(COUNTRY_ISO2).upper()}"
if PATHOGEN_NAME:
    freq_title += f" · {PATHOGEN_NAME}"

folium_h3_choropleth(
    cell_stats,
    "n_year_months",
    group_col="pathogen_name",
    title=freq_title,
)

## Duration map (`n_years`)

Fill color = **max** `n_years` across pathogens in the cell. Hover/click a hex for a horizontal stats table (one column per pathogen).

In [ ]:
dur_title = "n_years"
if COUNTRY_ISO2:
    dur_title += f" · {str(COUNTRY_ISO2).upper()}"
if PATHOGEN_NAME:
    dur_title += f" · {PATHOGEN_NAME}"

folium_h3_choropleth(
    cell_stats,
    "n_years",
    group_col="pathogen_name",
    title=dur_title,
)

## Top cells

Highest frequency and longest duration under the current filter (one row per `h3_cell` × `pathogen_name`).

In [ ]:
cols = [
    "h3_cell",
    "pathogen_name",
    "n_year_months",
    "n_years",
    "year_span",
    "n_records",
    "year_min",
    "year_max",
]

print("Top by n_year_months")
display(cell_stats[cols].head(15))

print("Top by n_years")
display(
    cell_stats.sort_values(
        ["n_years", "year_span", "n_year_months"],
        ascending=False,
    )[cols].head(15)
)